# Chapter 2: Streamlit — Turning Analysis Into a Shareable App

### Why This Matters

A brilliant analysis trapped in a Jupyter notebook only helps people comfortable running Python. **Streamlit** converts a plain Python script into an interactive, browser-based web application — with widgets, live filtering, and charts — using pure Python, no HTML/CSS/JavaScript required. This is precisely the tool that turns a one-off analysis into a self-service tool a manager can open and explore themselves.

In [ ]:
# app.py — Streamlit apps run as standalone Python scripts, not notebook cells: `streamlit run app.py`
import streamlit as st
import pandas as pd

st.title("Sales Explorer")

region = st.selectbox("Region", ["All", "East", "West", "South"])

df = pd.DataFrame({"region": ["East", "West", "South"], "sales": [120, 150, 90]})
if region != "All":
    df = df[df["region"] == region]

st.metric("Total Sales", f"{df['sales'].sum():,.0f}")
st.dataframe(df, use_container_width=True)

Streamlit's core mental model, worth understanding before writing a single widget: **the entire script re-runs top to bottom every time the user interacts with any widget** — selecting a new region in the dropdown re-executes the whole `app.py` file from the top, with `region` now holding the new value. This is fundamentally different from a notebook, where you execute cells individually and state persists between them — in a Streamlit app, there's no persistent "cell state" at all; the script's logic must be written to correctly reconstruct the right output every time it reruns from scratch, given whatever the current widget values are.

`st.selectbox`, `st.metric`, and `st.dataframe` are three of dozens of Streamlit **elements** — each one both displays something and, for input widgets, returns the user's current selection as a normal Python value you use immediately in the rest of your script.

### 2.1 Building a Realistic Dashboard

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.title("Regional Sales Performance Dashboard")

df = pd.DataFrame({
    "region": ["North","South","East","West"]*3,
    "category": ["Electronics","Apparel","Home"]*4,
    "sales": [125,98,112,145,130,101,120,150,99,140,133,152]
})

col1, col2 = st.columns(2)                                   # a two-column LAYOUT
region_filter = col1.multiselect("Region", df["region"].unique(), default=list(df["region"].unique()))
category_filter = col2.multiselect("Category", df["category"].unique(), default=list(df["category"].unique()))

filtered = df[df["region"].isin(region_filter) & df["category"].isin(category_filter)]

st.metric("Total Sales", f"₹{filtered['sales'].sum():,}")
st.bar_chart(filtered.groupby("region")["sales"].sum())
st.dataframe(filtered, use_container_width=True)

st.download_button("Download filtered data as CSV", filtered.to_csv(index=False), "filtered_sales.csv")

`st.columns(2)` lays widgets out side by side rather than stacked vertically — small layout tools like this are what separate a genuinely usable internal dashboard from a plain vertical list of controls. `st.download_button` closes the loop, letting the end user export exactly the filtered slice they've constructed interactively — directly useful the moment a manager wants to take a specific cut of the data into their own further analysis or a slide deck.

### 2.2 Caching — Essential for Real Applications

In [ ]:
@st.cache_data                          # caches the RESULT — re-runs only if the arguments change
def load_data():
    return pd.read_csv("large_sales_data.csv")

@st.cache_resource                      # caches an OBJECT that shouldn't be recreated each rerun — e.g. a loaded ML model
def load_model():
    import joblib
    return joblib.load("churn_model.joblib")

Recall the "entire script reruns on every interaction" behavior from Section 2's opening — without caching, an app that loads a large CSV or a trained model (Volume 05's `joblib.load`) would reload it from disk on *every single widget click*, making the app painfully slow. `@st.cache_data` (for data — DataFrames, arrays, anything you'd compare by value) and `@st.cache_resource` (for objects like database connections or loaded models — compared by identity, not value) are decorators (recall Volume 01, Chapter 13) that store the function's result after its first call and instantly return the cached result on subsequent reruns with the same inputs, rather than recomputing. Adding these two decorators to the right functions is, in practice, the single highest-leverage performance fix in almost any real Streamlit app.

### Cheat Sheet — Streamlit

| Task | Code |
|---|---|
| Run the app | `streamlit run app.py` (from the terminal, not inside a notebook) |
| Text/number input | `st.selectbox`, `st.slider`, `st.text_input`, `st.multiselect` |
| Display data | `st.dataframe(df)`, `st.metric(label, value)` |
| Charts | `st.bar_chart(df)`, `st.line_chart(df)`, or embed `st.pyplot(fig)` / `st.plotly_chart(fig)` |
| Layout | `col1, col2 = st.columns(2)` |
| Cache expensive data loads | `@st.cache_data` |
| Cache loaded models/connections | `@st.cache_resource` |
| Let users export | `st.download_button(label, data, filename)` |

---